In [ ]:
# import libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# load data
df_prod = pd.read_csv('../data/Global_production_quantity.csv')
df_species = pd.read_csv('../data/CL_FI_SPECIES_GROUPS.csv')
df_countries = pd.read_csv('../data/CL_FI_COUNTRY_GROUPS.csv')

In [ ]:
# preview production data
df_prod.info()

In [ ]:
# preview specie data
df_species.head()

In [ ]:
# preview country data
df_countries.head()

In [ ]:
# filter specie data to keep only seaweed species
mask_columns = ["CPC_Class_Es","CPC_Class_Ar","CPC_Class_Cn","CPC_Class_Ru", "ISSCAAP_Group_Cn", "ISSCAAP_Group_Ru", "CPC_Class_Fr","CPC_Group_Fr","CPC_Group_Es", "CPC_Group_Ar","CPC_Group_Cn","CPC_Group_Ru", "ISSCAAP_Group_Es","ISSCAAP_Group_Ar", "Yearbook_Group_Es", "Yearbook_Group_Ar", "Yearbook_Group_Cn", "Yearbook_Group_Ru", "ISSCAAP_Group_En"]
df_species_filter_alguae = df_species.drop(columns=mask_columns)[df_species["ISSCAAP_Group_Fr"].isin(["Algues rouges", "Algues brunes", "Algues vertes"])]

# get list of seaweed species
list_algae_species = df_species_filter_alguae["3A_Code"].tolist()

# filter production data to keep only seaweed species
df_prod_alguae = df_prod[df_prod["SPECIES.ALPHA_3_CODE"].isin(list_algae_species)]

# rename some columns
df_prod_alguae = df_prod_alguae.rename(columns={"PRODUCTION_SOURCE_DET.CODE": "source_production","COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production"})

# rename production origin (as Récolte or Culture)
df_prod_alguae["source_production"] = df_prod_alguae["source_production"].apply(lambda x: "Récolte" if x == "CAPTURE" else "Culture")


In [ ]:
# filter country data to keep relevant columns only
df_countries_filter = df_countries[["UN_Code", 'Name_Fr', "Continent_Group_Fr"]]

# fix French name of South Korea
mask_korea = df_countries_filter["Name_Fr"] == "République de Corée"
df_countries_filter.loc[mask_korea, "Name_Fr"] = "Corée du Sud"

# get list of european countries
df_europe = df_countries_filter[df_countries_filter["Continent_Group_Fr"] == "Europe"]
list_europe = df_europe["UN_Code"].tolist()

In [ ]:
# merge production and country data
df_prod_alguae_country = df_prod_alguae.merge(df_countries_filter, on="UN_Code", how="left")

## Plot production evolution over years

In [ ]:
# copy data to plot
data_fig1 = df_prod_alguae_country.copy()

# set country name or category of countries
mask_continent = data_fig1["Continent_Group_Fr"] == "Asie"
mask_countries = data_fig1["Name_Fr"].isin(["Indonésie", "Philippines", "Corée du Sud", "Chine"])
data_fig1["Pays"] = "Autres pays hors Asie"
data_fig1.loc[mask_continent, "Pays"] = "Autres pays d'Asie"
data_fig1.loc[mask_continent & mask_countries, "Pays"] = data_fig1.loc[mask_continent & mask_countries, "Name_Fr"]

# group data by year
data_fig1_by_year = data_fig1[["Année", "Pays", "Production"]].groupby(["Année", "Pays"]).sum().reset_index()

### PC version

In [ ]:
# set colors
colors = [('Chine','#1D3B6E'),('Indonésie','#209490'),('Corée du Sud','#5B8FCB'),('Philippines','#C3ADD4'), \
    ("Autres pays d'Asie", '#F6A01E'), ("Autres pays hors Asie",'#1E52A1')]

# plot production
fig1 = go.Figure()

for name_color_tuple in colors:
    fig1.add_trace(go.Scatter(
        x=data_fig1_by_year[data_fig1_by_year["Pays"]== name_color_tuple[0]]["Année"],
        y=data_fig1_by_year[data_fig1_by_year["Pays"]== name_color_tuple[0]]["Production"],
        mode='lines',
        line=dict(width=0.5, color=name_color_tuple[1]),
        stackgroup='one',
        fillcolor=name_color_tuple[1],
        name=name_color_tuple[0],
        hovertemplate="""Production : %{y}""" + \
        """<extra></extra>""",
        hoverlabel=dict(font=dict(family="Montserrat")),
        )
    )

fig1.update_xaxes(
    showgrid=False,
    ticklabelstandoff=10,
)

fig1.update_yaxes(
    title=dict(
        text="Production (tonnes d'algues)",
        font=dict(size=16)
    ),
    gridcolor="#1D3B6E",
    tickvals=[0, 10000000, 20000000, 30000000, 40000000],
    ticklabelstandoff=20,
)

fig1.update_layout(
    hovermode='x unified',
    font_family="Montserrat",
    #title_text="Evolution de la production d'algues<br>(en tonnes) par pays",
    #font_family="Parkinsans",
    #title_font_weight=800,
    #font_weight=600,
    font_color="#1D3B6E",
    paper_bgcolor ="#FDF2EE",
    plot_bgcolor ="#FDF2EE",
    legend=dict(
        orientation="h",
        font_family="Montserrat",
        font_color="#113972",
        bgcolor="#FDF2EE",
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5
    ),
    margin=dict(
        l=20,
        r=20,
        t=30,
        b=20
    ),
    height=400,
    width=900,
)

fig1.show()

In [ ]:
fig1.write_html("../figures/production_evolution_pc.html", include_plotlyjs="cdn")

### Mobile version

#### Graphe "camembert" repartition_culture_recolte Monde et Europe

In [ ]:
df_prod_alguae_country_fig_3 = df_prod_alguae_country.rename(columns={"Name_Fr": "Pays", "Continent_Group_Fr":"Continent"})

In [ ]:

colors= ['#209490','#5B8FCB' ]
df_prod_alguae_country_fig_3_europe = df_prod_alguae_country_fig_3[df_prod_alguae_country_fig_3["Continent"] == "Europe"]
data_go = df_prod_alguae_country_fig_3[["source_production", "Production"]].groupby("source_production").sum().reset_index()

fig_3 = go.Figure(data=[go.Pie(labels=data_go["source_production"],
                             values=data_go["Production"],
                               pull=[0.2,0],
                               texttemplate = "%{label} <br> %{percent:.0%}",
                               )])
fig_3.update_traces(hoverinfo='skip',textfont_size=14,
                  marker=dict(colors=colors), textposition='outside')
fig_3.update_layout(
    showlegend=False,
    title_text="Répartition entre récolte et<br>culture d'algues dans le monde",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
)

data_go_bis = df_prod_alguae_country_fig_3_europe[["source_production", "Production"]].groupby("source_production").sum().reset_index()

fig_3_bis = go.Figure(data=[go.Pie(labels=data_go_bis["source_production"],
                             values=data_go_bis["Production"],
                                   pull=[0.2, 0],
                                    texttemplate = "%{label} <br> %{percent:.0%}"
                               )])
fig_3_bis.update_traces(hoverinfo='skip', textfont_size=14,
                  marker=dict(colors=colors), textposition='outside')
fig_3_bis.update_layout(
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
    showlegend=False,
    title_text="Répartition entre récolte et<br>culture d'algues en Europe"
)
display(fig_3,fig_3_bis)
# fig_3
# fig_3 = go.Figure(data=[go.Pie(labels=["Hello", "Bonjour", "Salut"],
#                              values=[1,2,4],pull=[0.2,0]
#                                )])
# fig_3.update_traces(hoverinfo='skip', textinfo='label+percent', textfont_size=14,
#                   marker=dict(colors=colors), textposition='outside')

In [ ]:
fig_3.write_html("repartition_culture_recolte_monde_mobile.html", include_plotlyjs="cdn")
fig_3_bis.write_html("repartition_culture_recolte_europe_mobile.html", include_plotlyjs="cdn")

#### Graphe recolte_algues_monde_par_pays

In [ ]:
df_prod_alguae_country_fig_4 = df_prod_alguae_country.rename(columns={"Name_Fr": "Pays_2", "Continent_Group_Fr":"Continent"})
df_prod_alguae_country_fig_4_recolte = df_prod_alguae_country_fig_4[df_prod_alguae_country_fig_4["source_production"] == "Récolte"]

In [ ]:
def get_category_bis(row):
    if row["Pays_2"] in ["Chili", "Norvège", "Japon", "Inde", "Indonésie", "France"]:
       return row["Pays_2"]
    elif row["Pays_2"].lower() == "états-unis d'amérique":
      return "Etats-Unis"
    else:
      return "Autres pays"

Liste couleurs dispos : ['#1D3B6E','#5B8FCB', '#FDF2ED', '#209490', '#C73175', '#F6A01E', '#1E52A1', '#97C7EC', '#13716A', '#991358', '#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']

 '#0074E4' : FR
 '#1D3B6E' : Chine
 '#209490' : Indonésie
 '#C73175' : Norvège
 '#5B8FCB' : Corée du Sud
 '#97C7EC' : Philippines
 '#F6A01E' : Autres pays d'Asie
 '#1E52A1' : Autres pays hors Asie
 '#13716A' : Chili
 '#991358' : Inde
 '#E5800B' : Japon
 '#CDE8FA' : Etats-Unis
 '#3573B9' : Autres pays (du monde)
 '#58B7B0' : Irlande
 '#D85F9F' : Islande
 '#F9C06B' : Russie
 '#E1D3E9' : Royaume-Uni
 '#F18882' : Autres pays d'Europe

In [ ]:
df_prod_alguae_country_fig_4_recolte['Pays'] = df_prod_alguae_country_fig_4_recolte.apply(lambda row: get_category_bis(row), axis=1)
df_prod_alguae_country_fig_4_recolte_by_year = df_prod_alguae_country_fig_4_recolte[["Année", "Pays","Production"]].groupby(["Année", "Pays"]).sum().reset_index()
colors = [('Chili','#13716A'),('Norvège','#C73175'),("Indonésie",'#209490'),('Inde','#991358'), ('Japon', '#E5800B'), ('France','#0074E4'), ("Etats-Unis", '#CDE8FA'), ('Autres pays', '#3573B9')]

fig_4 = create_multi_traces_area_plot(df_prod_alguae_country_fig_4_recolte_by_year,colors, "Récolte d'algues dans le monde<br>(en tonnes) par pays")
fig_4

In [ ]:
fig_4.write_html("recolte_algues_monde_par_pays_mobile.html", include_plotlyjs="cdn")

#### Graphe production_algues_europe_par_pays

In [ ]:
def get_country_europe(row):
    if row["Pays_2"] in ["Norvège", "France", "Irlande", "Islande"]:
       return row["Pays_2"]
    elif row["Pays_2"] == "Fédération de Russie":
      return "Russie"
    elif row["Pays_2"] == "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord":
      return "Royaume-Uni"
    else:
      return "Autres pays d'Europe"

In [ ]:
df_prod_alguae_europe = df_prod_alguae[df_prod_alguae["UN_Code"].isin(list_europe)].rename(columns={"COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production"}).merge(df_countries_filter, on="UN_Code", how="left").rename(columns={"Name_Fr": "Pays_2", "Continent_Group_Fr": "Continent"})
df_prod_alguae_europe["Pays"] = df_prod_alguae_europe.apply(lambda row: get_country_europe(row), axis=1)
df_prod_alguae_europe_country_by_year = df_prod_alguae_europe[["Année", "Pays", "Production"]].groupby(["Année", "Pays"]).sum().reset_index()

colors = [('Norvège','#C73175'),('France','#0074E4'),('Irlande','#58B7B0'),('Islande','#D85F9F'), ("Russie",'#F9C06B'), ('Royaume-Uni','#E1D3E9'), ("Autres pays d'Europe",'#F18882')]

fig_5 = create_multi_traces_area_plot(df_prod_alguae_europe_country_by_year,colors, plot_title="Production d'algues en Europe<br>(en tonnes) par pays")
fig_5

In [ ]:
fig_5.write_html("production_algues_europe_par_pays_mobile.html", include_plotlyjs="cdn")

#### Graphe especes_algues_produites_europe

**Les données ici sont récupérées de ce Google sheet : https://docs.google.com/spreadsheets/d/1bhuzeFcvNo5S55Zbi_yOA3IuDdaUHtkU_QoQnPsOTC0/edit?gid=2037908735#gid=2037908735**

**La source originelle est ici : https://data.jrc.ec.europa.eu/collection/id-00363**

On crée le dataframe pour les espèces produites en Europe en culture

In [ ]:
list_algae_culture = ['Alaria sp.','Ascophyllum nodosum','Asparagopsis sp.','Brown seaweed','Calliblepharis jubata','Caulerpa sp.','Chondrus sp.','Codium sp.','Falkenbergia sp.','Fucus sp.','Gracilaria sp.','Gracilariopsis longissima','Himanthalia sp.','Kappaphycus','Laminaria sp.','Palmaria sp.','Porphyra sp.','Saccharina sp.','Schizymenia jonssonii','Ulva sp.','Ulvella lens','Undaria sp.','Espèce inconnue','Vertebrata lanosa']
list_nb_culture = [17,1,1,2,1,1,2,2,1,5,3,2,2,1,10,8,2,29,1,15,1,3,3,1]

In [ ]:
df_culture = pd.DataFrame({"algae":list_algae_culture, "nb_culture": list_nb_culture })
df_culture_sort = df_culture.sort_values(by="nb_culture", ascending=False)
df_culture_sort

On crée le dataframe pour les espèces produites en Europe en récolte

In [ ]:
list_algae_recolte = ['Alaria sp.','Ascophyllum nodosum','Asparagopsis sp.','Bifurcaria bifurcata','Brown seaweed','Calcareous algae','Chondrus sp.','Chorda filum','Coccotylus truncatus','Codium sp.','Corallina sp.','Cystoseira sp.','Delesseria sanguinea','Dilsea carnosa','Fucus sp.','Furcellaria lumbricalis','Gelidium sp.','Gigartina sp.','Gracilaria sp.','Grateloupia turuturu','Green seaweed','Halopteris scoparia','Himanthalia sp.','Laminaria sp.','Lithothamnium calcareum','Mastocarpus stellatus','Osmundea pinnatifida','Padina pavonica','Palmaria sp.','Pelvetia sp.','Petalonia binghamiae','Porphyra sp.','Pterocladiella capillacea','Red seaweed','Saccharina sp.','Salicornia sp.','Sargassum sp.','Solieria sp.','Ulva sp.','Undaria sp.','Espèce inconnue','Vertebrata lanosa','Zonaria tournefortii']
list_nb_recolte =[13,25,2,1,7,1,21,1,1,6,1,1,1,1,30,3,3,3,1,1,2,1,30,37,4,3,5,1,36,1,1,25,1,3,26,2,2,1,32,22,10,4,1]

In [ ]:
df_autre_recolte = pd.DataFrame({"algae":list_algae_recolte, "nb_recolte": list_nb_recolte })
df_autre_recolte

In [ ]:
df_autre_recolte_sort = df_autre_recolte.sort_values(by="nb_recolte", ascending=False)
df_autre_recolte_sort

In [ ]:
colors_recolte = ['#1D3B6E','#5B8FCB', '#FDF2ED', '#209490', '#C73175', '#F6A01E', '#1E52A1', '#97C7EC', '#13716A', '#991358', '#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']
colors_culture = ['#F6A01E','#E5800B', '#FDF2ED', '#1D3B6E', '#5B8FCB','#209490' , '#3573B9', '#13716A','#F9C06B', '#1E52A1','#D85F9F', '#C73175', '#58B7B0', '#991358', '#CDE8FA','#97C7EC'  , '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']

In [ ]:

## Récolte
fig_6 = go.Figure()
fig_6.add_trace(go.Treemap(
    labels = df_autre_recolte_sort["algae"],
    parents = ["",] *15,
    values =  df_autre_recolte_sort["nb_recolte"],
    texttemplate="""<span style="font-weight:800">%{label}</span> <br>""" + \
    """%{value} entreprises (%{percentEntry:.1%})""",
    marker_colors= colors_recolte,
    root_color="white"

))
fig_6.update_layout(
    title_text="Espèces d'algues produites<br>en Europe (Récolte)",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
  )
fig_6.update_traces(marker=dict(cornerradius=5), hoverinfo="skip")

## Culture
fig_6_bis = go.Figure()
fig_6_bis.add_trace(go.Treemap(
    labels = df_culture_sort["algae"],
    parents = ["",] *15,
    # parents = ["",] *24,
    values = df_culture_sort["nb_culture"],
     texttemplate="""<span style="font-weight:800">%{label}</span> <br>""" + \
    """%{value} entreprises (%{percentEntry:.1%})""",
    marker_colors= colors_culture,
    root_color="white",

))
fig_6_bis.update_layout(
    title_text="Espèces d'algues produites<br>en Europe (Culture)",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
  )

fig_6_bis.update_traces(marker=dict(cornerradius=5), hoverinfo="skip")
display(
    fig_6,
    fig_6_bis
)

In [ ]:
fig_6.write_html("especes_algues_produites_europe_recolte_mobile.html", include_plotlyjs="cdn")

In [ ]:
fig_6_bis.write_html("especes_algues_produites_europe_culture_mobile.html", include_plotlyjs="cdn")

#### Graphe projection_emplois_2030

**La source des données est issue de ce rapport : https://phyconomy.net/wp-content/uploads/2020/10/Seaweed_for_Europe-Hidden_Champion_of_the_ocean-Report.pdf**

In [ ]:

x=['Emplois<br>directs', 'Emplois<br>indirects', 'Emplois<br>induits', 'Total<br>Emplois']
df =pd.DataFrame({"x":x, "y":[0, 37, 65, 0], "z":[22,17,11,50], "a": [15, 11, 7, 33] })
# df

fig_7 = go.Figure(go.Bar(x=df["x"], y=df["y"], name='',marker_color='white')
    )
fig_7.add_trace(go.Bar(x=df["x"], y=df["z"], name='Produits cultivés et finis en Europe', marker_color='#F6A01E', text=df["z"], textfont=dict(color="white",size=10)) )
fig_7.add_trace(go.Bar(x=df["x"], y=df["a"], name='Produits cultivés hors Europe, finis en Europe',marker_color='#209490', text=df["a"], textfont=dict(color="white", size=10)))

fig_7.update_layout(barmode='stack',
                  legend_orientation='h',
                  paper_bgcolor = 'white',  # Fully transparent background
                  plot_bgcolor = 'white',   # Fully transparent plot area
                  title_text="Emplois potentiels créés<br>(milliers de ETP) en Europe<br>par l'industrie des algues en 2030",
                  font_family="Parkinsans",
                  title_font_weight=800,
                  font_weight=600,
                  font_color="#1D3B6E",
    )

fig_7.update_yaxes(showline=False, visible=False)
fig_7.update_xaxes(showline=True, linewidth=2, linecolor='#1D3B6E', tickfont=dict(size=10))
fig_7.update_traces(hoverinfo="skip", marker=dict(
                              line=dict(width=1,
                                        color='rgba(0,0,0,0)'),
                              ), textposition='inside'
)
fig_7.show()

In [ ]:
fig_7.write_html("projection_emplois_2030_mobile.html", include_plotlyjs="cdn")

#### Graphe chiffre_affaires_europe_par_pays

**Les données ici sont issues du google Sheet suivant : https://docs.google.com/spreadsheets/d/15DgqS1Lia5SoO2-rK6bCOh41MroGS3ZJo90b-S0PWXo/edit?pli=1&gid=1772352892#gid=1772352892**

**La source originelle est ici : https://data.jrc.ec.europa.eu/collection/id-00363**

In [ ]:
df_ca_alguae = pd.read_csv('../data/Algae-industry-Europe-socioeconomic - Socio-economic data.csv')
df_ca_alguae.info()

In [ ]:
df_ca_alguae.rename(columns={"Organism group": "production_type", "Average available turnover in the last 5 years": "turnover_l5y"}).head(10)

On renomme certaines colonnes

In [ ]:
df_ca_alguae_bis = df_ca_alguae.rename(columns={"Organism group": "production_type", "Average available turnover in the last 5 years": "turnover_l5y", "Is Algae considered the main business stream": "is_algae_business", "% of business focused on algae - ESTIMATED": "algae_business_percent_estimated"})

On filtre les lignes en visant ls entreprises centrés sur les macro-algues uniqument et si leur business des algues est bien leur business principale

In [ ]:
df_ca_alguae_macro_clean = df_ca_alguae_bis[["ID", "Country", "production_type", "turnover_l5y", "is_algae_business"]]
df_ca_alguae_macro_clean.head(10)

In [ ]:
df_ca_alguae_macro_clean_by_country = df_ca_alguae_macro_clean[(df_ca_alguae_macro_clean['is_algae_business'] == "Yes")&(df_ca_alguae_macro_clean['production_type'] == "Macroalgae")]
df_ca_alguae_macro_clean_by_country["turnover_l5y"] = df_ca_alguae_macro_clean_by_country["turnover_l5y"].str.replace(',', "").fillna(0)
df_ca_alguae_macro_clean_by_country["turnover_l5y"] = df_ca_alguae_macro_clean_by_country["turnover_l5y"].astype('int64')
df_ca_alguae_macro_clean_by_country = df_ca_alguae_macro_clean_by_country[["Country", "turnover_l5y"]].rename(columns={"turnover_l5y": "Chiffre_d_affaires", "Country": "Pays"}).groupby(by="Pays").sum().reset_index().sort_values(by="Chiffre_d_affaires", ascending=False)
df_ca_alguae_macro_clean_by_country

In [ ]:
df_ca_alguae_macro_by_country = df_ca_alguae_macro_clean_by_country[df_ca_alguae_macro_clean_by_country["Chiffre_d_affaires"] > 0]

 '#0074E4' : FR
 '#1D3B6E' : Chine
 '#209490' : Indonésie
 '#C73175' : Norvège
 '#5B8FCB' : Corée du Sud
 '#C3ADD4' : Philippines
 '#F6A01E' : Autres pays d'Asie
 '#1E52A1' : Autres pays hors Asie
 '#13716A' : Chili
 '#991358' : Inde
 '#E5800B' : Japon
 '#CDE8FA' : Etats-Unis
 '#3573B9' : Autres pays (du monde)
 '#58B7B0' : Irlande
 '#D85F9F' : Islande
 '#F9C06B' : Russie
 '#E1D3E9' : Royaume-Uni
 '#F18882' : Autres pays d'Europe

In [ ]:
def translate_country(row):
  if (row["Pays"] == "Ireland"):
    return 'Irlande'
  elif (row["Pays"] == "Norway"):
    return 'Norvège'
  elif (row["Pays"] == "Iceland"):
    return 'Islande'
  elif (row["Pays"] == "Spain"):
    return 'Espagne'
  elif (row["Pays"] == "Estonia"):
    return 'Estonie'
  elif (row["Pays"] == "UK"):
    return 'Royaume-Uni'
  elif (row["Pays"] == "Denmark"):
    return 'Danemark'
  elif (row["Pays"] == "Sweden"):
    return 'Suède'
  else:
    return row["Pays"]

In [ ]:
colors= ['#0074E4','#58B7B0', '#C73175', '#D85F9F', '#991358','#F6A01E', '#1E52A1', '#E1D3E9', '#13716A','#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', '#F8C2BB']
df_ca_alguae_macro_by_country["Pays"] = df_ca_alguae_macro_by_country.apply(translate_country, axis=1)
fig_10 = go.Figure(
    data=[
        go.Bar(x=df_ca_alguae_macro_by_country["Pays"],
               y=df_ca_alguae_macro_by_country["Chiffre_d_affaires"],
               marker_color=colors,
               hovertemplate="""<span style='font-family:Parkinsans; font-weight:600'>Pays : %{x}</span><br>""" + \
               """<span style='font-family:Parkinsans; font-weight:600'>Chiffre d'affaires : %{y}</span>""" + \
      """<extra></extra>""",
        )
    ],
    layout=dict(
        barcornerradius=15,
    ),
)
fig_10.update_layout(
    title_text="Chiffre d’affaires réalisé (en euros)<br>par pays en Europe grâce à la<br>production d’algues",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
    paper_bgcolor = 'white',  # Fully transparent background
    plot_bgcolor = 'white',   # Fully transparent plot area

  )
fig_10.show()

In [ ]:
fig_10.write_html("chiffre_affaires_europe_par_pays_mobile.html", include_plotlyjs="cdn")